# 01 - Inspection des données

> Notebook de cadrage qualité et de préparation initiale des données du projet eau.

Ce notebook documente l'inspection des fichiers bruts, les anomalies détectées, les règles de nettoyage retenues et les contraintes qui doivent être propagées dans le pipeline.

## Objectif et sommaire

> Objectif : fiabiliser les données sources avant toute modélisation, export CSV ou restitution BI.

Lecture recommandée :

1. chargement et premier aperçu des sources ;
2. fonctions d'inspection et audit ciblé de la qualité ;
3. préparation analytique et normalisation des indicateurs ;
4. contexte DWFA et glossaire métier ;
5. construction de la table dashboard et des scores ;
6. exports CSV pour le pipeline ;
7. insights et trame narrative du dashboard ;
8. conclusion pipeline.

<a id="RNCP37837BC04"></a>

### Installation de Python et des bibliothèques nécessaires

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Bloc compétence :</strong> <span style="background-color: #FFF3E0; padding: 2px 6px; border-radius: 3px;">RNCP37837BC04</span> — Organiser un projet data · Gérer la documentation
</div>

In [39]:
# Import des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date

# Configuration de l'affichage
pd.set_option('display.max_columns', None)  # Afficher toutes les colonnes
pd.set_option('display.max_rows', 100)      # Limiter à 100 lignes
pd.set_option('display.width', None)        # Pas de limite de largeur

# Style des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("✅ Bibliothèques chargées avec succès !")

✅ Bibliothèques chargées avec succès !


<a id="RNCP37837BC02-COLLECT"></a>

## Partie 1 - Chargement des données

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Compétence :</strong> <span style="background-color: #E8F4F8; padding: 2px 6px; border-radius: 3px;">Identifier et collecter</span> — Utiliser des outils adaptés pour préparer les données
</div>

In [40]:
# Définir les chemins vers les données
import importlib
import src.schema_fr as schema_fr
importlib.reload(schema_fr)

paths = schema_fr.resolve_project_paths()
PROJECT_ROOT = paths['PROJECT_ROOT']
RAW_DATA_PATH = paths['RAW_DATA_PATH']
PROCESSED_DATA_PATH = paths['PROCESSED_DATA_PATH']

# Chargement des fichiers bruts
population_source = pd.read_csv(RAW_DATA_PATH / "Population.csv", sep=",")
zone_source = pd.read_csv(RAW_DATA_PATH / "RegionCountry.csv", sep=",")
stabilite_politique_source = pd.read_csv(RAW_DATA_PATH / "PoliticalStability.csv", sep=",")
mortalite_eau_source = pd.read_csv(RAW_DATA_PATH / "MortalityRateAttributedToWater.csv", sep=",")
services_eau_source = pd.read_csv(RAW_DATA_PATH / "BasicAndSafelyManagedDrinkingWaterServices.csv", sep=",")

# Variables de travail temporaires avant normalisation canonique
population = population_source.copy()
zone = zone_source.copy()
stabilite_politique = stabilite_politique_source.copy()
mortalite_eau = mortalite_eau_source.copy()
services_eau = services_eau_source.copy()

print("✅ Fichiers chargés avec succès !")
print(f"   - Dossier source : {RAW_DATA_PATH}")
print(f"   - population : {len(population_source):,} lignes")
print(f"   - zone : {len(zone_source):,} lignes")
print(f"   - stabilite_politique : {len(stabilite_politique_source):,} lignes")
print(f"   - mortalite_eau : {len(mortalite_eau_source):,} lignes")
print(f"   - services_eau : {len(services_eau_source):,} lignes")

✅ Fichiers chargés avec succès !
   - Dossier source : C:\Users\feria\Documents\P10\data\raw
   - population : 20,914 lignes
   - zone : 194 lignes
   - stabilite_politique : 3,526 lignes
   - mortalite_eau : 549 lignes
   - services_eau : 10,476 lignes


In [41]:
# Contrat de schéma partagé : source brute, schéma canonique français, présentation notebook et export
from src.schema_fr import (
    COLONNES_NOTEBOOK_FR,
    EXPORT_PBI_FR,
    SCHEMAS_SOURCES_FR,
    VALEURS_NOTEBOOK_FR,
    normaliser_table_source,
    vue_notebook_fr,
 )

population = normaliser_table_source(globals().get("population_source", population), "population")
zone = normaliser_table_source(globals().get("zone_source", zone), "zone")
stabilite_politique = normaliser_table_source(globals().get("stabilite_politique_source", stabilite_politique), "stabilite_politique")
mortalite_eau = normaliser_table_source(globals().get("mortalite_eau_source", mortalite_eau), "mortalite_eau")
services_eau = normaliser_table_source(globals().get("services_eau_source", services_eau), "services_eau")

def afficher_notebook(df, lignes=10):
    display(vue_notebook_fr(df).head(lignes))

def inspecter_dataframe_fr(df, nom="jeu_de_donnees"):
    inspecter_dataframe(vue_notebook_fr(df), nom=nom)

print("✅ Schéma canonique français appliqué aux tables sources.")
print(f"   - population : {list(population.columns)}")
print(f"   - zone : {list(zone.columns)}")
print(f"   - stabilite_politique : {list(stabilite_politique.columns)}")

✅ Schéma canonique français appliqué aux tables sources.
   - population : ['pays', 'granularite', 'annee', 'population']
   - zone : ['region', 'pays']
   - stabilite_politique : ['pays', 'annee', 'stabilite_politique', 'granularite']


<a id="RNCP37837BC02-EXPLORE"></a>

## Partie 2 - Premier aperçu des fichiers

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Compétence :</strong> <span style="background-color: #E8F4F8; padding: 2px 6px; border-radius: 3px;">Explorer et pré-traiter</span> — Utiliser des langages/outils pour comprendre les caractéristiques des données
</div>

In [42]:
# Aperçu des premières lignes de chaque fichier
print("=" * 60)
print("POPULATION - Premières lignes")
print("=" * 60)
display(vue_notebook_fr(population).head(3))

POPULATION - Premières lignes


,pays,granularite,annee,population
0,Afghanistan,Total,2000,20779.953
1,Afghanistan,Hommes,2000,10689.508
2,Afghanistan,Femmes,2000,10090.449


In [43]:
print("=" * 60)
print("ZONE - Premières lignes")
print("=" * 60)
display(vue_notebook_fr(zone).head(3))

ZONE - Premières lignes


,region,pays
0,Europe,Albania
1,Europe,Andorra
2,Europe,Armenia


In [44]:
print("=" * 60)
print("STABILITÉ POLITIQUE - Premières lignes")
print("=" * 60)
display(vue_notebook_fr(stabilite_politique).head(3))

STABILITÉ POLITIQUE - Premières lignes


,pays,annee,stabilite_politique,granularite
0,Afghanistan,2000,-2.44,Total
1,Afghanistan,2002,-2.04,Total
2,Afghanistan,2003,-2.20,Total


In [45]:
print("=" * 60)
print("MORTALITÉ EAU - Premières lignes")
print("=" * 60)
display(vue_notebook_fr(mortalite_eau).head(3))

MORTALITÉ EAU - Premières lignes


,annee,pays,granularite,mortalite_wash_services_non_surs,deces_wash
0,2016,Afghanistan,Femmes,15.31193,NaN
1,2016,Afghanistan,Hommes,12.61297,NaN
2,2016,Afghanistan,Total,13.92067,4824.353


In [46]:
print("=" * 60)
print("SERVICES EAU - Premières lignes")
print("=" * 60)
display(vue_notebook_fr(services_eau).head(3))

SERVICES EAU - Premières lignes


,annee,pays,granularite,acces_eau_potable_base_pct,acces_eau_potable_gere_securise_pct
0,2000,Afghanistan,Rural,21.61913,NaN
1,2000,Afghanistan,Total,27.77190,NaN
2,2000,Afghanistan,Urbain,49.48745,NaN


<a id="RNCP37837BC02-INSPECT"></a>

## Partie 3 - Fonctions d'inspection

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Compétence :</strong> <span style="background-color: #E8F4F8; padding: 2px 6px; border-radius: 3px;">Vérifier la cohérence</span> — Préparer et vérifier la fiabilité des données structurées
</div>

### 3.1 Fonction de base

In [47]:
def inspector(df):

    """
    Génère un rapport d'inspection complet pour un DataFrame.

    Paramètres:
    -----------
    df : pandas.DataFrame
        Le DataFrame à inspecter

    Retourne:
    ---------
    None (affiche le rapport dans le terminal)
    """

    # Debut de l'inspection
    print("#####     DEBUT DE L'INSPECTION     #####")

    # Structure du DataFrame avec df.shape
    print("\n----------------------")
    print("Structure du DataFrame:")
    print(f'Nb de lignes: {df.shape[0]} ; Nb de colonnes: {df.shape[1]}')

    # Types de données avec df.dtypes
    print("\n----------------------")
    print("Types de données:")
    print(df.dtypes)

    # Présence d'une potentielle clé unique (identifiant) avec df.nunique()
    print("\n----------------------")
    print("Nombre de valeurs uniques (pour identifiant PK / FK):")
    print(df.nunique())

    # Présence de valeurs manquantes avec df.isnull().sum()
    print("\n----------------------")
    print("Valeurs manquantes:")
    print(df.isnull().sum())

    # Fin de l'inspection
    print("\n#####     FIN DE L'INSPECTION     #####")

In [48]:
# Test de la fonction d'inspection sur la vue française du jeu de données population
inspector(vue_notebook_fr(population))

#####     DEBUT DE L'INSPECTION     #####

----------------------
Structure du DataFrame:
Nb de lignes: 20914 ; Nb de colonnes: 4

----------------------
Types de données:
pays            object
granularite     object
annee            int64
population     float64
dtype: object

----------------------
Nombre de valeurs uniques (pour identifiant PK / FK):
pays             239
granularite        5
annee             19
population     20515
dtype: int64

----------------------
Valeurs manquantes:
pays           0
granularite    0
annee          0
population     0
dtype: int64

#####     FIN DE L'INSPECTION     #####


In [49]:
# Inspection de base des fichiers avec une présentation francisée
import sys
from pathlib import Path

src_path = PROJECT_ROOT / "src" if "PROJECT_ROOT" in globals() else None
if src_path is None:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent
    src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

if "population" not in globals() or "population_source" not in globals():
    RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
    population_source = pd.read_csv(RAW_DATA_PATH / "Population.csv", sep=",")
    zone_source = pd.read_csv(RAW_DATA_PATH / "RegionCountry.csv", sep=",")
    stabilite_politique_source = pd.read_csv(RAW_DATA_PATH / "PoliticalStability.csv", sep=",")
    mortalite_eau_source = pd.read_csv(RAW_DATA_PATH / "MortalityRateAttributedToWater.csv", sep=",")
    services_eau_source = pd.read_csv(RAW_DATA_PATH / "BasicAndSafelyManagedDrinkingWaterServices.csv", sep=",")

    population = normaliser_table_source(population_source, "population")
    zone = normaliser_table_source(zone_source, "zone")
    stabilite_politique = normaliser_table_source(stabilite_politique_source, "stabilite_politique")
    mortalite_eau = normaliser_table_source(mortalite_eau_source, "mortalite_eau")
    services_eau = normaliser_table_source(services_eau_source, "services_eau")

from data_manager import inspecter_dataframe

inspecter_dataframe_fr(population, nom="population")
inspecter_dataframe_fr(zone, nom="zone")
inspecter_dataframe_fr(stabilite_politique, nom="stabilite_politique")
inspecter_dataframe_fr(mortalite_eau, nom="mortalite_eau")
inspecter_dataframe_fr(services_eau, nom="services_eau")


📊 RAPPORT D'INSPECTION : POPULATION

📐 DIMENSIONS
   - Nombre de lignes   : 20,914
   - Nombre de colonnes : 4

📋 COLONNES ET TYPES
----------------------------------------
   pays                      → object
   granularite               → object
   annee                     → int64
   population                → float64

❓ VALEURS MANQUANTES (NA)
----------------------------------------
   ✅ Aucune valeur manquante !

🔑 RECHERCHE DE CLÉ UNIQUE (IDENTIFIANT)
----------------------------------------
   ❌ Aucune colonne n'est une clé unique
   💡 Les colonnes avec le plus de valeurs uniques :
      - pays: 239 valeurs uniques
      - granularite: 5 valeurs uniques
      - annee: 19 valeurs uniques

📈 STATISTIQUES DESCRIPTIVES
----------------------------------------

   Colonnes numériques :


,annee,population
count,20914.00,20914.00
mean,2009.05,22531.64
std,5.48,100016.85
min,2000.00,0.00
25%,2004.00,348.35
50%,2009.00,3016.34
75%,2014.00,11150.43
max,2018.00,1459377.61



   Colonnes catégorielles :


,pays,granularite
count,20914,20914
unique,239,5
top,Afghanistan,Total
freq,95,4430



Fin du rapport pour population


📊 RAPPORT D'INSPECTION : ZONE

📐 DIMENSIONS
   - Nombre de lignes   : 194
   - Nombre de colonnes : 2

📋 COLONNES ET TYPES
----------------------------------------
   region                    → object
   pays                      → object

❓ VALEURS MANQUANTES (NA)
----------------------------------------
   ✅ Aucune valeur manquante !

🔑 RECHERCHE DE CLÉ UNIQUE (IDENTIFIANT)
----------------------------------------
   ✅ 'pays' est une clé unique (194 valeurs uniques)

📈 STATISTIQUES DESCRIPTIVES
----------------------------------------

   Colonnes catégorielles :


...


Fin du rapport pour mortalite_eau


📊 RAPPORT D'INSPECTION : SERVICES_EAU

📐 DIMENSIONS
   - Nombre de lignes   : 10,476
   - Nombre de colonnes : 5

📋 COLONNES ET TYPES
----------------------------------------
   annee                     → int64
   pays                      → object
   granularite               → object
   acces_eau_potable_base_pct → float64
   acces_eau_potable_gere_securise_pct → float64

❓ VALEURS MANQUANTES (NA)
----------------------------------------
   ⚠️  Total de valeurs manquantes : 8,251
   - acces_eau_potable_base_pct : 1,061 (10.1%)
   - acces_eau_potable_gere_securise_pct : 7,190 (68.6%)

🔑 RECHERCHE DE CLÉ UNIQUE (IDENTIFIANT)
----------------------------------------
   ❌ Aucune colonne n'est une clé unique
   💡 Les colonnes avec le plus de valeurs uniques :
      - annee: 18 valeurs uniques
      - pays: 194 valeurs uniques
      - granularite: 3 valeurs uniques

📈 STATISTIQUES DESCRIPTIVES
----------------------------------------

   Colonnes numér

,annee,acces_eau_potable_base_pct,acces_eau_potable_gere_securise_pct
count,10476.00,9415.00,3286.00
mean,2008.50,83.96,66.07
std,5.19,19.97,30.38
min,2000.00,4.08,0.00
25%,2004.00,75.93,41.90
50%,2008.50,93.12,73.97
75%,2013.00,98.95,94.78
max,2017.00,100.00,100.00



   Colonnes catégorielles :


,pays,granularite
count,10476,10476
unique,194,3
top,Afghanistan,Rural
freq,54,3492



Fin du rapport pour services_eau



### 3.2 Fonction avancée

In [50]:
def inspecter_dataframe(df, nom="DataFrame"):
    """
    Génère un rapport d'inspection complet pour un DataFrame.

    Paramètres:
    -----------
    df : pandas.DataFrame
        Le DataFrame à inspecter
    nom : str
        Le nom du DataFrame (pour l'affichage)

    Retourne:
    ---------
    None (affiche le rapport dans le terminal)
    """

    df = vue_notebook_fr(df)

    # Séparateur visuel
    separateur = "=" * 70

    # === EN-TÊTE ===
    print(f"\n{separateur}")
    print(f"📊 RAPPORT D'INSPECTION : {nom.upper()}")
    print(f"{separateur}\n")

    # === 1. DIMENSIONS ===
    print(f"📐 DIMENSIONS")
    print(f"   - Nombre de lignes   : {df.shape[0]:,}")
    print(f"   - Nombre de colonnes : {df.shape[1]}")
    print()

    # === 2. TYPES DE DONNÉES ===
    print(f"📋 COLONNES ET TYPES")
    print("-" * 40)
    for col in df.columns:
        print(f"   {col:25} → {df[col].dtype}")
    print()

    # === 3. VALEURS MANQUANTES ===
    print(f"❓ VALEURS MANQUANTES (NA)")
    print("-" * 40)
    na_counts = df.isna().sum()
    na_total = na_counts.sum()

    if na_total == 0:
        print("   ✅ Aucune valeur manquante !")
    else:
        print(f"   ⚠️  Total de valeurs manquantes : {na_total:,}")
        for col in df.columns:
            if na_counts[col] > 0:
                pct = (na_counts[col] / len(df)) * 100
                print(f"   - {col:25} : {na_counts[col]:,} ({pct:.1f}%)")
    print()

    # === 4. RECHERCHE DE CLÉ UNIQUE ===
    print(f"🔑 RECHERCHE DE CLÉ UNIQUE (IDENTIFIANT)")
    print("-" * 40)
    cle_trouvee = False

    for col in df.columns:
        nb_valeurs_uniques = df[col].nunique()
        nb_lignes = len(df)

        # Une colonne est une clé unique si chaque valeur est différente
        if nb_valeurs_uniques == nb_lignes:
            print(f"   ✅ '{col}' est une clé unique ({nb_valeurs_uniques:,} valeurs uniques)")
            cle_trouvee = True
        elif nb_valeurs_uniques == nb_lignes - df[col].isna().sum():
            # Cas où il y a des NA mais les valeurs non-NA sont uniques
            print(f"   ⚠️  '{col}' pourrait être une clé (unique hors NA)")
            cle_trouvee = True

    if not cle_trouvee:
        print("   ❌ Aucune colonne n'est une clé unique")
        print("   💡 Les colonnes avec le plus de valeurs uniques :")
        # Afficher les 3 colonnes avec le plus de valeurs uniques
        for col in df.columns[:3]:
            print(f"      - {col}: {df[col].nunique():,} valeurs uniques")
    print()

    # === 5. STATISTIQUES DESCRIPTIVES ===
    print(f"📈 STATISTIQUES DESCRIPTIVES")
    print("-" * 40)

    # Colonnes numériques
    cols_numeriques = df.select_dtypes(include=[np.number]).columns.tolist()
    if cols_numeriques:
        print("\n   Colonnes numériques :")
        display(df[cols_numeriques].describe().round(2))

    # Colonnes catégorielles (texte)
    cols_categorielles = df.select_dtypes(include=['object']).columns.tolist()
    if cols_categorielles:
        print("\n   Colonnes catégorielles :")
        display(df[cols_categorielles].describe())

    print(f"\n{separateur}")
    print(f"Fin du rapport pour {nom}")
    print(f"{separateur}\n")

<a id="RNCP37837BC02-AUDIT"></a>

## Partie 4 - Audit ciblé de la qualité des données

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Compétence :</strong> <span style="background-color: #E8F4F8; padding: 2px 6px; border-radius: 3px;">Vérifier la cohérence · Analyse temporelle</span> — Identifier les anomalies bloquantes avant le pipeline
</div>

> Objectif : identifier les anomalies bloquantes avant toute préparation pour le pipeline et le dashboard.

In [51]:
# Audit rapide des clés de jointure, des granularités et des unités
datasets = {
    "population": population,
    "zone": zone,
    "stabilite_politique": stabilite_politique,
    "mortalite_eau": mortalite_eau,
    "services_eau": services_eau,
}

for nom, df in datasets.items():
    print("=" * 90)
    print(f"AUDIT : {nom.upper()}")
    print("=" * 90)
    print(f"Dimensions : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
    print(f"Colonnes   : {list(df.columns)}")
    if "granularite" in df.columns:
        print(f"Granularités observées : {sorted(df['granularite'].dropna().unique().tolist())}")
    if "pays" in df.columns:
        print(f"Pays uniques : {df['pays'].nunique():,}")
    if "annee" in df.columns:
        print(f"Période : {int(df['annee'].min())} -> {int(df['annee'].max())}")
    print()

population_country_set = set(population["pays"].dropna().unique())
zone_country_set = set(zone["pays"].dropna().unique())
countries_without_region = sorted(population_country_set - zone_country_set)

print("=" * 90)
print("PAYS DE POPULATION SANS REGION ASSOCIEE")
print("=" * 90)
print(f"Pays non rattachés : {len(countries_without_region)} / {len(population_country_set)} ({len(countries_without_region) / len(population_country_set):.1%})")
print(countries_without_region[:15])

print()
print("=" * 90)
print("CONTROLE DE L'UNITE DE POPULATION")
print("=" * 90)
sample_population = population.query("pays == 'China' and granularite == 'Total' and annee == 2018")
display(vue_notebook_fr(sample_population))
print("Interprétation attendue : la valeur est en milliers d'habitants et devra être multipliée par 1000.")

AUDIT : POPULATION
Dimensions : 20,914 lignes x 4 colonnes
Colonnes   : ['pays', 'granularite', 'annee', 'population']
Granularités observées : ['Female', 'Male', 'Rural', 'Total', 'Urban']
Pays uniques : 239
Période : 2000 -> 2018

AUDIT : ZONE
Dimensions : 194 lignes x 2 colonnes
Colonnes   : ['region', 'pays']
Pays uniques : 194

AUDIT : STABILITE_POLITIQUE
Dimensions : 3,526 lignes x 4 colonnes
Colonnes   : ['pays', 'annee', 'stabilite_politique', 'granularite']
Granularités observées : ['Total']
Pays uniques : 200
Période : 2000 -> 2018

AUDIT : MORTALITE_EAU
Dimensions : 549 lignes x 5 colonnes
Colonnes   : ['annee', 'pays', 'granularite', 'mortalite_wash_services_non_surs', 'deces_wash']
Granularités observées : ['Female', 'Male', 'Total']
Pays uniques : 183
Période : 2016 -> 2016

AUDIT : SERVICES_EAU
Dimensions : 10,476 lignes x 5 colonnes
Colonnes   : ['annee', 'pays', 'granularite', 'acces_eau_potable_base_pct', 'acces_eau_potable_gere_securise_pct']
Granularités observées :

,pays,granularite,annee,population
3876,China,Total,2018,1459377.612


Interprétation attendue : la valeur est en milliers d'habitants et devra être multipliée par 1000.


### 4.1 Fonctions utilitaires de nettoyage

 > On standardise les noms de pays et on prépare les règles de rattachement région afin de sécuriser les jointures.

In [52]:
import unicodedata

def normaliser_texte(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    text = " ".join(text.split())
    return text

COUNTRY_RENAMES = {
    "North Macedonia": "Republic of North Macedonia",
}

REGION_OVERRIDES = {
    "American Samoa": "Western Pacific",
    "Anguilla": "Americas",
    "Aruba": "Americas",
    "Bermuda": "Americas",
    "Bonaire, Sint Eustatius and Saba": "Americas",
    "British Virgin Islands": "Americas",
    "Cayman Islands": "Americas",
    "Channel Islands": "Europe",
    "China, Hong Kong SAR": "Western Pacific",
    "China, Macao SAR": "Western Pacific",
    "China, Taiwan Province of": "Western Pacific",
    "China, mainland": "Western Pacific",
    "Curaçao": "Americas",
    "Falkland Islands (Malvinas)": "Americas",
    "Faroe Islands": "Europe",
    "French Guyana": "Americas",
    "French Polynesia": "Western Pacific",
    "Gibraltar": "Europe",
    "Greenland": "Americas",
    "Guadeloupe": "Americas",
    "Guam": "Western Pacific",
    "Holy See": "Europe",
    "Isle of Man": "Europe",
    "Liechtenstein": "Europe",
    "Martinique": "Americas",
    "Mayotte": "Africa",
    "Montserrat": "Americas",
    "Netherlands Antilles (former)": "Americas",
    "New Caledonia": "Western Pacific",
    "Northern Mariana Islands": "Western Pacific",
    "Palestine": "Eastern Mediterranean",
    "Puerto Rico": "Americas",
    "Réunion": "Africa",
    "Saint Barthélemy": "Americas",
    "Saint Helena, Ascension and Tristan da Cunha": "Africa",
    "Saint Pierre and Miquelon": "Americas",
    "Saint-Martin (French part)": "Americas",
    "Serbia and Montenegro": "Europe",
    "Sint Maarten (Dutch part)": "Americas",
    "Sint Maarten  (Dutch part)": "Americas",
    "Sudan (former)": "Eastern Mediterranean",
    "Tokelau": "Western Pacific",
    "Turks and Caicos Islands": "Americas",
    "United States Virgin Islands": "Americas",
    "Wallis and Futuna Islands": "Western Pacific",
    "Western Sahara": "Africa",
}

def standardiser_pays(series):
    cleaned = series.map(normaliser_texte)
    return cleaned.replace(COUNTRY_RENAMES)

def min_max_scale(series):
    numeric_series = pd.to_numeric(series, errors="coerce")
    if numeric_series.dropna().empty:
        return pd.Series(np.nan, index=series.index, dtype="float64")
    min_value = numeric_series.min()
    max_value = numeric_series.max()
    if pd.isna(min_value) or pd.isna(max_value) or min_value == max_value:
        return pd.Series(0.0, index=series.index, dtype="float64")
    return (numeric_series - min_value) / (max_value - min_value)

### 4.2 Construction du référentiel pays-régions

 > On combine la table de référence fournie et des overrides documentés pour couvrir les territoires absents.

#### 4.2.1 Règle métier retenue

 > La nomenclature de référence suit les macro-régions du fichier source (Africa, Americas, Eastern Mediterranean, Europe, South-East Asia, Western Pacific), pas des continents stricts.

In [53]:
zone_reference = zone.rename(columns={"pays": "country_raw"}).copy()
zone_reference["country_clean"] = standardiser_pays(zone_reference["country_raw"])

manual_region_reference = pd.DataFrame(
    [{"country_clean": country, "region": region} for country, region in REGION_OVERRIDES.items()]
 )

country_region_reference = pd.concat(
    [
        zone_reference[["country_clean", "region"]],
        manual_region_reference,
    ],
    ignore_index=True,
).dropna(subset=["country_clean"]).drop_duplicates(subset=["country_clean"], keep="first")

population_country_reference = pd.DataFrame(
    {"country_clean": sorted(standardiser_pays(population["pays"]).dropna().unique())}
).merge(country_region_reference, on="country_clean", how="left")

countries_without_region_after_override = population_country_reference[
    population_country_reference["region"].isna()
].sort_values("country_clean")

print(f"Pays de population couverts par une région : {population_country_reference['region'].notna().sum():,} / {len(population_country_reference):,}")
print(f"Pays restant sans région : {len(countries_without_region_after_override)}")
afficher_notebook(population_country_reference)

if not countries_without_region_after_override.empty:
    afficher_notebook(countries_without_region_after_override, lignes=len(countries_without_region_after_override))

Pays de population couverts par une région : 239 / 239
Pays restant sans région : 0


,pays_normalise,region
0,Afghanistan,Eastern Mediterranean
1,Albania,Europe
2,Algeria,Africa
3,American Samoa,Western Pacific
4,Andorra,Europe
5,Angola,Africa
6,Anguilla,Americas
7,Antigua and Barbuda,Americas
8,Argentina,Americas
9,Armenia,Europe


## Partie 5 - Préparation analytique pour la priorisation des pays

> Attention méthodologique : l'indicateur WASH n'isole pas l'eau potable seule. Dans ce notebook, il est traité comme un proxy de vulnérabilité sanitaire liée à l'eau et à l'assainissement, à croiser avec les indicateurs d'accès à l'eau potable et de stabilité politique.

In [54]:
def preparer_base_indicateur(df, country_col="pays", year_col="annee", granularity_col="granularite"):
    base = df.copy()
    base["country"] = standardiser_pays(base[country_col]) if country_col in base.columns else pd.NA
    base["year"] = pd.to_numeric(base[year_col], errors="coerce").astype("Int64") if year_col in base.columns else pd.NA
    base["granularity"] = base[granularity_col].map(normaliser_texte) if granularity_col in base.columns else "Total"
    base = base.merge(country_region_reference, left_on="country", right_on="country_clean", how="left")
    return base.drop(columns=["country_clean"], errors="ignore")

def construire_table_indicateur(base_df, value_map, source_name, scope_note):
    frames = []
    for source_col, meta in value_map.items():
        temp = base_df[["country", "region", "year", "granularity", source_col]].copy()
        temp = temp.rename(columns={source_col: "value"})
        temp["indicator"] = meta["indicator"]
        temp["unit"] = meta["unit"]
        temp["source"] = source_name
        temp["scope_note"] = scope_note
        temp["value"] = pd.to_numeric(temp["value"], errors="coerce")
        frames.append(temp)
    return pd.concat(frames, ignore_index=True)

def extraire_indicateur(df, indicator_name, granularity_name, value_name):
    subset = df[(df["indicator"] == indicator_name) & (df["granularity"] == granularity_name)].copy()
    subset = subset[["country", "region", "year", "value"]].rename(columns={"value": value_name})
    return subset

def fusionner_pays_annee(left, right):
    merged = left.merge(right, on=["country", "year"], how="outer", suffixes=("", "_new"))
    if "region_new" in merged.columns:
        merged["region"] = merged["region"].fillna(merged["region_new"])
        merged = merged.drop(columns=["region_new"])
    return merged

def calculer_score_pondere(df, weighted_columns):
    numerator = pd.Series(0.0, index=df.index)
    denominator = pd.Series(0.0, index=df.index)
    for column_name, weight in weighted_columns:
        valid_mask = df[column_name].notna()
        numerator = numerator + df[column_name].fillna(0) * weight
        denominator = denominator + valid_mask.astype(float) * weight
    return pd.Series(np.where(denominator > 0, numerator / denominator * 100, np.nan), index=df.index)

### 5.1 Construction de la table longue normalisée

 > Chaque source est harmonisée dans un format commun : pays, région, année, granularité, indicateur, valeur, unité, source et note de portée.

In [55]:
population_base = preparer_base_indicateur(population)
population_base["population"] = pd.to_numeric(population_base["population"], errors="coerce") * 1000

population_long = construire_table_indicateur(
    population_base,
    {
        "population": {"indicator": "population_people", "unit": "people"},
    },
    source_name="Population.csv",
    scope_note="Contexte démographique. Valeur convertie en habitants.",
)

stabilite_base = preparer_base_indicateur(stabilite_politique)
stabilite_long = construire_table_indicateur(
    stabilite_base,
    {
        "stabilite_politique": {"indicator": "political_stability", "unit": "index"},
    },
    source_name="PoliticalStability.csv",
    scope_note="Indicateur institutionnel.",
)

services_base = preparer_base_indicateur(services_eau)
services_long = construire_table_indicateur(
    services_base,
    {
        "acces_eau_potable_base_pct": {
            "indicator": "basic_drinking_water_pct",
            "unit": "percent",
        },
        "acces_eau_potable_gere_securise_pct": {
            "indicator": "safely_managed_drinking_water_pct",
            "unit": "percent",
        },
    },
    source_name="BasicAndSafelyManagedDrinkingWaterServices.csv",
    scope_note="Accès à l'eau potable, directement exploitable pour la création et la modernisation des services.",
)

mortalite_base = preparer_base_indicateur(mortalite_eau)
mortalite_long = construire_table_indicateur(
    mortalite_base,
    {
        "mortalite_wash_services_non_surs": {
            "indicator": "wash_mortality_rate_per_100k",
            "unit": "per_100k",
        },
        "deces_wash": {"indicator": "wash_deaths", "unit": "people"},
    },
    source_name="MortalityRateAttributedToWater.csv",
    scope_note="Proxy sanitaire WASH, disponible uniquement pour 2016. A utiliser comme référence transversale et non comme série temporelle.",
)

normalized_long = pd.concat(
    [population_long, stabilite_long, services_long, mortalite_long],
    ignore_index=True,
)

normalized_long = normalized_long.dropna(subset=["country", "year", "indicator"]).sort_values(
    ["country", "year", "indicator", "granularity"]
).reset_index(drop=True)

print(f"Table longue normalisée : {len(normalized_long):,} lignes")
afficher_notebook(normalized_long)

Table longue normalisée : 46,490 lignes


,pays,region,annee,granularite,valeur,indicateur,unite,source,note_portee
0,Afghanistan,Eastern Mediterranean,2000,Rural,2.161913e+01,acces_eau_potable_pct,percent,BasicAndSafelyManagedDrinkingWaterServices.csv,"Accès à l'eau potable, directement exploitable..."
1,Afghanistan,Eastern Mediterranean,2000,Total,2.777190e+01,acces_eau_potable_pct,percent,BasicAndSafelyManagedDrinkingWaterServices.csv,"Accès à l'eau potable, directement exploitable..."
2,Afghanistan,Eastern Mediterranean,2000,Urbain,4.948745e+01,acces_eau_potable_pct,percent,BasicAndSafelyManagedDrinkingWaterServices.csv,"Accès à l'eau potable, directement exploitable..."
3,Afghanistan,Eastern Mediterranean,2000,Total,-2.440000e+00,stabilite_politique,index,PoliticalStability.csv,Indicateur institutionnel.
4,Afghanistan,Eastern Mediterranean,2000,Femmes,1.009045e+07,population_habitants,people,Population.csv,Contexte démographique. Valeur convertie en ha...
5,Afghanistan,Eastern Mediterranean,2000,Hommes,1.068951e+07,population_habitants,people,Population.csv,Contexte démographique. Valeur convertie en ha...
6,Afghanistan,Eastern Mediterranean,2000,Rural,1.565747e+07,population_habitants,people,Population.csv,Contexte démographique. Valeur convertie en ha...
7,Afghanistan,Eastern Mediterranean,2000,Total,2.077995e+07,population_habitants,people,Population.csv,Contexte démographique. Valeur convertie en ha...
8,Afghanistan,Eastern Mediterranean,2000,Urbain,4.436282e+06,population_habitants,people,Population.csv,Contexte démographique. Valeur convertie en ha...
9,Afghanistan,Eastern Mediterranean,2000,Rural,NaN,service_eau_gere_securise_pct,percent,BasicAndSafelyManagedDrinkingWaterServices.csv,"Accès à l'eau potable, directement exploitable..."


In [56]:
duplicate_mask = normalized_long.duplicated(subset=["country", "year", "granularity", "indicator"], keep=False)
duplicates_long = normalized_long[duplicate_mask].sort_values(
    ["country", "year", "indicator", "granularity"]
).copy()

indicator_quality = (
    normalized_long.groupby(["indicator", "granularity"], dropna=False)
    .agg(
        rows=("value", "size"),
        non_null_values=("value", lambda series: series.notna().sum()),
        countries=("country", "nunique"),
        years=("year", "nunique"),
        missing_regions=("region", lambda series: series.isna().sum()),
    )
    .reset_index()
    .sort_values(["indicator", "granularity"])
    .reset_index(drop=True)
)

print(f"Doublons pays-année-granularité-indicateur : {len(duplicates_long):,}")
print(f"Lignes avec région manquante              : {normalized_long['region'].isna().sum():,}")
afficher_notebook(indicator_quality)

if not duplicates_long.empty:
    afficher_notebook(duplicates_long, lignes=20)

Doublons pays-année-granularité-indicateur : 0
Lignes avec région manquante              : 0


,indicateur,granularite,lignes,valeurs_non_nulles,nb_pays,nb_annees,regions_manquantes
0,acces_eau_potable_pct,Rural,3492,2953,194,18,0
1,acces_eau_potable_pct,Total,3492,3449,194,18,0
2,acces_eau_potable_pct,Urbain,3492,3013,194,18,0
3,stabilite_politique,Total,3526,3526,200,18,0
4,population_habitants,Femmes,3828,3828,205,19,0
5,population_habitants,Hommes,3828,3828,205,19,0
6,population_habitants,Rural,4414,4414,237,19,0
7,population_habitants,Total,4430,4430,239,19,0
8,population_habitants,Urbain,4414,4414,237,19,0
9,service_eau_gere_securise_pct,Rural,3492,612,194,18,0


In [57]:
latest_available_year = int(normalized_long["year"].dropna().max())
coverage_latest_year = (
    normalized_long[normalized_long["year"] == latest_available_year]
    .groupby(["indicator", "granularity"], dropna=False)["country"]
    .nunique()
    .reset_index(name="countries_in_latest_year")
    .sort_values(["indicator", "granularity"])
    .reset_index(drop=True)
)

mortality_years = sorted(
    normalized_long[normalized_long["indicator"].isin(["wash_mortality_rate_per_100k", "wash_deaths"])]
    ["year"].dropna().astype(int).unique().tolist()
)

print(f"Dernière année observable dans la table longue : {latest_available_year}")
print(f"Années disponibles pour la mortalité WASH : {mortality_years}")
if mortality_years == [2016]:
    print("Contrainte validée : la table mortalité sert de référence ponctuelle 2016 pour le pipeline.")
display(coverage_latest_year)

Dernière année observable dans la table longue : 2018
Années disponibles pour la mortalité WASH : [2016]
Contrainte validée : la table mortalité sert de référence ponctuelle 2016 pour le pipeline.


,indicator,granularity,countries_in_latest_year
0,political_stability,Total,198
1,population_people,Female,203
2,population_people,Male,203
3,population_people,Rural,235
4,population_people,Total,237
5,population_people,Urban,235


### Conclusion sur la table mortalité

> La table de mortalité WASH ne contient qu'une seule année de référence, 2016. Dans le pipeline, elle doit donc être traitée comme une couche de contexte ou de scoring ponctuel, et non comme une source de tendance temporelle. Les analyses d'évolution dans le temps doivent s'appuyer d'abord sur la population, l'accès à l'eau potable et la stabilité politique.

## Partie 6 - Contexte DWFA et glossaire des donnees

 > Cette section ajoute le contexte metier necessaire avant la construction du dataset dashboard. Elle explicite la logique DWFA, les definitions des indicateurs et leurs limites d'interpretation.

### 6.1 Lecture attendue

 > Le glossaire ci-dessous distingue les donnees sources, les champs calcules du pipeline et la relation de chaque indicateur a la vulnerabilite.

In [58]:
# Chargement du glossaire projet et rappel du cadre DWFA
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'data').exists() else cwd.parent
GLOSSARY_PATH = PROJECT_ROOT / 'data' / 'processed' / 'csv_enrichis' / 'en' / 'glossaire_indicateurs_dwfa.csv'

glossaire_dwfa = pd.read_csv(GLOSSARY_PATH, sep=';')

dwfa_context = pd.DataFrame(
    [
        {
            'axe_dwfa': 'Vulnerabilite sanitaire',
            'lecture_projet': 'La mortalite WASH signale une exposition a des conditions d eau, d assainissement et d hygiene non sures.',
        },
        {
            'axe_dwfa': 'Capacite de service',
            'lecture_projet': 'Les indicateurs basic et safely managed mesurent le niveau et la qualite des services d eau potable.',
        },
        {
            'axe_dwfa': 'Structure territoriale',
            'lecture_projet': 'La part rurale et la part urbaine changent la difficulte de creation des infrastructures.',
        },
        {
            'axe_dwfa': 'Capacite institutionnelle',
            'lecture_projet': 'La stabilite politique renseigne sur le contexte de faisabilite des politiques publiques et du consulting.',
        },
        {
            'axe_dwfa': 'Limite majeure',
            'lecture_projet': 'La mortalite WASH n est disponible que pour 2016 et doit etre traitee comme un snapshot de reference.',
        },
    ]
)

display(dwfa_context)
display(glossaire_dwfa[['variable_projet', 'titre', 'unite_de_mesure', 'source_originale', 'relation_vulnerabilite']])

,axe_dwfa,lecture_projet
0,Vulnerabilite sanitaire,La mortalite WASH signale une exposition a des...
1,Capacite de service,Les indicateurs basic et safely managed mesure...
2,Structure territoriale,La part rurale et la part urbaine changent la ...
3,Capacite institutionnelle,La stabilite politique renseigne sur le contex...
4,Limite majeure,La mortalite WASH n est disponible que pour 20...


,variable_projet,titre,unite_de_mesure,source_originale,relation_vulnerabilite
0,political_stability,Stabilite politique et absence de violence,Indice,Banque mondiale - Worldwide Governance Indicators,-
1,population_total_people,Population totale,Personnes,United Nations DESA Population Division,+
2,population_rural_people,Population rurale,Personnes,United Nations DESA Population Division,+
3,rural_share_pct,Part de population rurale,Pourcentage,Champ calcule projet,+
4,urban_share_pct,Part de population urbaine,Pourcentage,Champ calcule projet,-
5,basic_drinking_water_pct,Population utilisant au moins un service basiq...,Pourcentage,WHO UNICEF JMP,-
6,safely_managed_drinking_water_pct,Population utilisant un service d'eau potable ...,Pourcentage,WHO UNICEF JMP,-
7,basic_minus_safe_gap_pct,Ecart entre service basique et service safely ...,Points de pourcentage,Champ calcule projet,+
8,wash_mortality_rate_per_100k,Taux de mortalite attribue aux services WASH n...,Deces pour 100000 habitants,Source sanitaire internationale utilisee dans ...,+
9,wash_deaths,Nombre de deces attribues aux services WASH no...,Personnes,Source sanitaire internationale utilisee dans ...,+


### 6.2 Conclusion contextuelle

 > Dans ce projet, les donnees eau ne sont pas lues isolément. Elles sont interpretees dans un cadre DWFA simplifie qui relie exposition sanitaire, capacite de service, structure territoriale et capacite institutionnelle. Ce cadrage justifie l'usage combine de la mortalite WASH, de l'acces a l'eau potable, de la population et de la stabilite politique dans la priorisation.

## Partie 7 - Construction de la table dashboard et des scores d'action

 > On construit une vue pays-année large, puis on dérive trois scores métier : création de services, modernisation des services et gouvernance publique.

In [59]:
population_total = extraire_indicateur(normalized_long, "population_people", "Total", "population_total_people")
population_rural = extraire_indicateur(normalized_long, "population_people", "Rural", "population_rural_people")
stability_total = extraire_indicateur(normalized_long, "political_stability", "Total", "political_stability")
basic_water_total = extraire_indicateur(normalized_long, "basic_drinking_water_pct", "Total", "basic_drinking_water_pct")
safe_water_total = extraire_indicateur(normalized_long, "safely_managed_drinking_water_pct", "Total", "safely_managed_drinking_water_pct")
wash_rate_total = extraire_indicateur(normalized_long, "wash_mortality_rate_per_100k", "Total", "wash_mortality_rate_per_100k")
wash_deaths_total = extraire_indicateur(normalized_long, "wash_deaths", "Total", "wash_deaths")

dashboard_water = population_total.copy()
for frame in [population_rural, stability_total, basic_water_total, safe_water_total, wash_rate_total, wash_deaths_total]:
    dashboard_water = fusionner_pays_annee(dashboard_water, frame)

dashboard_water = dashboard_water.sort_values(["country", "year"]).reset_index(drop=True)

print(f"Table dashboard pays-année : {len(dashboard_water):,} lignes")
afficher_notebook(dashboard_water)

Table dashboard pays-année : 4,466 lignes


,pays,region,annee,population_totale,population_rurale,political_stability,acces_eau_potable_pct,service_eau_gere_securise_pct,mortalite_wash_pour_100k,deces_wash
0,Afghanistan,Eastern Mediterranean,2000,20779953.0,15657474.0,-2.44,27.77190,NaN,NaN,NaN
1,Afghanistan,Eastern Mediterranean,2001,21606988.0,16318324.0,NaN,27.79726,NaN,NaN,NaN
2,Afghanistan,Eastern Mediterranean,2002,22600770.0,17086910.0,-2.04,29.90076,NaN,NaN,NaN
3,Afghanistan,Eastern Mediterranean,2003,23680871.0,17909063.0,-2.20,32.00507,NaN,NaN,NaN
4,Afghanistan,Eastern Mediterranean,2004,24726684.0,18692107.0,-2.30,34.12623,NaN,NaN,NaN
5,Afghanistan,Eastern Mediterranean,2005,25654277.0,19378962.0,-2.07,36.26526,NaN,NaN,NaN
6,Afghanistan,Eastern Mediterranean,2006,26433049.0,19961972.0,-2.22,38.40636,NaN,NaN,NaN
7,Afghanistan,Eastern Mediterranean,2007,27100536.0,20464923.0,-2.41,40.84418,NaN,NaN,NaN
8,Afghanistan,Eastern Mediterranean,2008,27722276.0,20929119.0,-2.69,43.31506,NaN,NaN,NaN
9,Afghanistan,Eastern Mediterranean,2009,28394813.0,21415593.0,-2.71,45.81909,NaN,NaN,NaN


In [60]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "data").exists() else cwd.parent

if "dashboard_water" in globals():
    dashboard_audit = dashboard_water.copy()
else:
    dashboard_audit = pd.read_csv(
        project_root / "data" / "processed" / "csv_enrichis" / "en" / "dashboard_water_country_year.csv",
        sep=";",
    )

china_overlap_countries = [
    "China",
    "China, mainland",
    "China, Hong Kong SAR",
    "China, Macao SAR",
    "China, Taiwan Province of",
]

china_overlap = (
    dashboard_audit.loc[
        dashboard_audit["country"].isin(china_overlap_countries),
        ["country", "year", "population_total_people"],
    ]
    .pivot(index="year", columns="country", values="population_total_people")
    .assign(
        components_sum=lambda df_: (
            df_["China, mainland"]
            + df_["China, Hong Kong SAR"]
            + df_["China, Macao SAR"]
            + df_["China, Taiwan Province of"]
        ),
        delta_china_minus_components=lambda df_: (
            df_["China"] - df_["components_sum"]
        ),
    )
    .reset_index()
    .sort_values("year")
)

overlap_detected = china_overlap["delta_china_minus_components"].abs().eq(0).all()

if overlap_detected:
    print(
        "Alerte audit population : China est deja egal a China, mainland + Hong Kong + Macao + Taiwan sur toutes les annees. "
        "Toute somme naive sur ces entites cree un double comptage."
    )
else:
    print("Aucun recouvrement parfait detecte entre China et ses composantes.")

afficher_notebook(
    china_overlap[
        [
            "year",
            "China",
            "China, mainland",
            "China, Hong Kong SAR",
            "China, Macao SAR",
            "China, Taiwan Province of",
            "components_sum",
            "delta_china_minus_components",
        ]
    ]
)
print(f"Delta max absolu : {china_overlap['delta_china_minus_components'].abs().max():,.0f}")

Alerte audit population : China est deja egal a China, mainland + Hong Kong + Macao + Taiwan sur toutes les annees. Toute somme naive sur ces entites cree un double comptage.


country,annee,China,"China, mainland","China, Hong Kong SAR","China, Macao SAR","China, Taiwan Province of",somme_composantes,ecart_chine_moins_composantes
0,2000,1.319551e+09,1.290551e+09,6606327.0,427782.0,21966527.0,1.319551e+09,0.0
1,2001,1.328341e+09,1.299130e+09,6664772.0,437938.0,22108713.0,1.328341e+09,0.0
2,2002,1.336765e+09,1.307352e+09,6701775.0,448821.0,22262299.0,1.336765e+09,0.0
3,2003,1.344908e+09,1.315304e+09,6724677.0,460165.0,22419792.0,1.344908e+09,0.0
4,2004,1.352871e+09,1.323085e+09,6744566.0,471597.0,22570224.0,1.352871e+09,0.0
5,2005,1.360735e+09,1.330776e+09,6769574.0,482858.0,22705713.0,1.360735e+09,0.0
6,2006,1.368528e+09,1.338409e+09,6802080.0,493799.0,22823848.0,1.368528e+09,0.0
7,2007,1.376266e+09,1.345994e+09,6840015.0,504511.0,22927215.0,1.376266e+09,0.0
8,2008,1.383986e+09,1.353569e+09,6881863.0,515239.0,23019045.0,1.383986e+09,0.0
9,2009,1.391725e+09,1.361169e+09,6924642.0,526400.0,23104546.0,1.391725e+09,0.0


Delta max absolu : 0


### 7.1 Scores d'action et segmentation

 > Le WASH est utilisé comme proxy de vulnérabilité. Les scores combinent donc accès à l'eau potable, vulnérabilité sanitaire, ruralité et stabilité politique.

In [ ]:
dashboard_water["rural_share_pct"] = (
    dashboard_water["population_rural_people"] / dashboard_water["population_total_people"]
    * 100
)
dashboard_water["basic_minus_safe_gap_pct"] = (
    dashboard_water["basic_drinking_water_pct"] - dashboard_water["safely_managed_drinking_water_pct"]
)

dashboard_water["risk_low_basic"] = 1 - min_max_scale(dashboard_water["basic_drinking_water_pct"])
dashboard_water["risk_low_safe"] = 1 - min_max_scale(dashboard_water["safely_managed_drinking_water_pct"])
dashboard_water["risk_high_gap"] = min_max_scale(dashboard_water["basic_minus_safe_gap_pct"])
dashboard_water["risk_high_mortality"] = min_max_scale(dashboard_water["wash_mortality_rate_per_100k"])
dashboard_water["risk_low_stability"] = 1 - min_max_scale(dashboard_water["political_stability"])
dashboard_water["risk_high_rurality"] = min_max_scale(dashboard_water["rural_share_pct"])

dashboard_water["wash_score_inverse"] = 1 - min_max_scale(
    dashboard_water["wash_mortality_rate_per_100k"]
)
dashboard_water["water_access_score"] = dashboard_water["basic_drinking_water_pct"] / 100
dashboard_water["gov_effectiveness_score"] = (
    0.5 * dashboard_water["wash_score_inverse"]
    + 0.5 * dashboard_water["water_access_score"]
)

dashboard_water["score_creation_services"] = calculer_score_pondere(
    dashboard_water,
    [
        ("risk_low_basic", 0.45),
        ("risk_high_rurality", 0.30),
        ("risk_high_mortality", 0.25),
    ],
)

dashboard_water["score_modernisation_services"] = calculer_score_pondere(
    dashboard_water,
    [
        ("risk_low_safe", 0.45),
        ("risk_high_gap", 0.35),
        ("risk_high_mortality", 0.20),
    ],
)

dashboard_water["score_gouvernance"] = dashboard_water["gov_effectiveness_score"] * 100

score_labels = {
    "score_creation_services": "Créer ou étendre les services d'eau",
    "score_modernisation_services": "Moderniser les services d'eau",
    "score_gouvernance": "Renforcer la gouvernance politique",
}

score_columns = list(score_labels.keys())
valid_score_mask = dashboard_water[score_columns].notna().any(axis=1)
dashboard_water["action_prioritaire"] = pd.NA
dashboard_water.loc[valid_score_mask, "action_prioritaire"] = (
    dashboard_water.loc[valid_score_mask, score_columns]
    .idxmax(axis=1)
    .map(score_labels)
)

afficher_notebook(
    dashboard_water[
        [
            "country",
            "region",
            "year",
            "basic_drinking_water_pct",
            "safely_managed_drinking_water_pct",
            "wash_mortality_rate_per_100k",
            "political_stability",
            "gov_effectiveness_score",
            "score_creation_services",
            "score_modernisation_services",
            "score_gouvernance",
            "action_prioritaire",
        ]
    ]
)

,pays,region,annee,acces_eau_potable_pct,service_eau_gere_securise_pct,mortalite_wash_pour_100k,political_stability,score_creation_services,score_modernisation_services,score_gouvernance,action_prioritaire
0,Afghanistan,Eastern Mediterranean,2000,27.77190,NaN,NaN,-2.44,81.041379,NaN,83.522727,Renforcer la gouvernance politique
1,Afghanistan,Eastern Mediterranean,2001,27.79726,NaN,NaN,NaN,81.086879,NaN,NaN,Créer ou étendre les services d'eau
2,Afghanistan,Eastern Mediterranean,2002,29.90076,NaN,NaN,-2.04,79.563971,NaN,75.946970,Créer ou étendre les services d'eau
3,Afghanistan,Eastern Mediterranean,2003,32.00507,NaN,NaN,-2.20,78.019713,NaN,78.977273,Renforcer la gouvernance politique
4,Afghanistan,Eastern Mediterranean,2004,34.12623,NaN,NaN,-2.30,76.442657,NaN,80.871212,Renforcer la gouvernance politique
5,Afghanistan,Eastern Mediterranean,2005,36.26526,NaN,NaN,-2.07,74.843526,NaN,76.515152,Renforcer la gouvernance politique
6,Afghanistan,Eastern Mediterranean,2006,38.40636,NaN,NaN,-2.22,73.256137,NaN,79.356061,Renforcer la gouvernance politique
7,Afghanistan,Eastern Mediterranean,2007,40.84418,NaN,NaN,-2.41,71.455582,NaN,82.954545,Renforcer la gouvernance politique
8,Afghanistan,Eastern Mediterranean,2008,43.31506,NaN,NaN,-2.69,69.625110,NaN,88.257576,Renforcer la gouvernance politique
9,Afghanistan,Eastern Mediterranean,2009,45.81909,NaN,NaN,-2.71,67.749659,NaN,88.636364,Renforcer la gouvernance politique


In [62]:
core_year_sets = [
    set(
        normalized_long[
            (normalized_long["indicator"] == indicator_name)
            & (normalized_long["granularity"] == "Total")
            & normalized_long["value"].notna()
        ]["year"].astype(int).unique()
    )
    for indicator_name in [
        "basic_drinking_water_pct",
        "political_stability",
        "wash_mortality_rate_per_100k",
    ]
]

analysis_year = max(set.intersection(*core_year_sets))
priority_snapshot = dashboard_water[dashboard_water["year"] == analysis_year].copy()
priority_snapshot = priority_snapshot[priority_snapshot["action_prioritaire"].notna()].copy()
priority_snapshot_focus = priority_snapshot[priority_snapshot["population_total_people"] >= 500000].copy()

print(f"Année commune retenue pour la priorisation : {analysis_year}")
print(f"Pays éligibles à la priorisation            : {len(priority_snapshot):,}")
print(f"Pays retenus après filtre population >= 500k : {len(priority_snapshot_focus):,}")

afficher_notebook(
    priority_snapshot_focus.sort_values("score_creation_services", ascending=False)[
        ["country", "region", "population_total_people", "score_creation_services", "action_prioritaire"]
    ]
)

afficher_notebook(
    priority_snapshot_focus.sort_values("score_modernisation_services", ascending=False)[
        ["country", "region", "population_total_people", "score_modernisation_services", "action_prioritaire"]
    ]
)

afficher_notebook(
    priority_snapshot_focus.sort_values("score_gouvernance", ascending=False)[
        ["country", "region", "population_total_people", "score_gouvernance", "action_prioritaire"]
    ]
)

Année commune retenue pour la priorisation : 2016
Pays éligibles à la priorisation            : 235
Pays retenus après filtre population >= 500k : 173


,pays,region,population_totale,score_creation_services,action_prioritaire
765,Chad,Africa,14561660.0,80.033254,Moderniser les services d'eau
3767,South Sudan,Africa,10832518.0,73.638902,Renforcer la gouvernance politique
2901,Niger,Africa,20788798.0,68.454585,Moderniser les services d'eau
746,Central African Republic,Africa,4537686.0,66.620769,Moderniser les services d'eau
1362,Ethiopia,Africa,103603462.0,65.857546,Moderniser les services d'eau
3729,Somalia,Eastern Mediterranean,14185636.0,64.300440,Moderniser les services d'eau
1305,Eritrea,Africa,3376557.0,62.707321,Créer ou étendre les services d'eau
632,Burundi,Africa,10487995.0,62.480822,Renforcer la gouvernance politique
1134,Democratic Republic of the Congo,Africa,78789127.0,61.988033,Renforcer la gouvernance politique
613,Burkina Faso,Africa,18646357.0,60.757072,Créer ou étendre les services d'eau


,pays,region,population_totale,score_modernisation_services,action_prioritaire
765,Chad,Africa,14561660.0,100.000000,Moderniser les services d'eau
3729,Somalia,Eastern Mediterranean,14185636.0,85.680581,Moderniser les services d'eau
3626,Sierra Leone,Africa,7328834.0,85.269905,Moderniser les services d'eau
746,Central African Republic,Africa,4537686.0,81.264262,Moderniser les services d'eau
2920,Nigeria,Africa,185960241.0,77.924590,Moderniser les services d'eau
2198,Lao People's Democratic Republic,Western Pacific,6845846.0,76.067516,Moderniser les services d'eau
4159,Uganda,Africa,39649166.0,71.701163,Moderniser les services d'eau
2787,Nepal,South-East Asia,27263433.0,70.815968,Moderniser les services d'eau
2901,Niger,Africa,20788798.0,70.082044,Moderniser les services d'eau
2445,Mali,Africa,17965443.0,69.989444,Moderniser les services d'eau


,pays,region,population_totale,score_gouvernance,action_prioritaire
3729,Somalia,Eastern Mediterranean,14185636.0,83.384953,Moderniser les services d'eau
765,Chad,Africa,14561660.0,76.207386,Moderniser les services d'eau
3767,South Sudan,Africa,10832518.0,75.451584,Renforcer la gouvernance politique
746,Central African Republic,Africa,4537686.0,74.981674,Moderniser les services d'eau
3053,Palestine,Eastern Mediterranean,4635654.0,74.810606,Renforcer la gouvernance politique
2920,Nigeria,Africa,185960241.0,73.612828,Moderniser les services d'eau
1134,Democratic Republic of the Congo,Africa,78789127.0,71.892094,Renforcer la gouvernance politique
632,Burundi,Africa,10487995.0,70.909687,Renforcer la gouvernance politique
2445,Mali,Africa,17965443.0,68.741307,Moderniser les services d'eau
1362,Ethiopia,Africa,103603462.0,65.631080,Moderniser les services d'eau


In [72]:
snapshot_audit = dashboard_water[dashboard_water['year'] == analysis_year].copy()

arbitrage_priorisation = pd.DataFrame(
    [
        {
            'theme': 'valeurs_manquantes_snapshot_2016',
            'constat': f"{int(snapshot_audit['action_prioritaire'].isna().sum())} pays non classes sur {len(snapshot_audit)} observations 2016",
            'decision': 'pas d\'imputation automatique ; exclusion des pays sans action prioritaire',
        },
        {
            'theme': 'couverture_service_gere_securise_2016',
            'constat': f"{int(snapshot_audit['safely_managed_drinking_water_pct'].isna().sum())} valeurs manquantes sur le service gere en toute securite",
            'decision': 'le score de modernisation reste calculable seulement quand les variables requises sont presentes',
        },
        {
            'theme': 'audit_population_china',
            'constat': 'China est egal a la somme China mainland + Hong Kong + Macao + Taiwan sur toutes les annees',
            'decision': 'conserver l\'information dans les marts mais interdire toute somme simultanee parent + composantes',
        },
        {
            'theme': 'normalisation_libelles_exports_pbi',
            'constat': 'les exports PBI normalisent China mainland en Chine, China Taiwan Province of en Taiwan, China Hong Kong SAR en Hong Kong et China Macao SAR en Macao',
            'decision': 'centraliser cette regle dans le script de reconstruction geographique pour eviter des corrections manuelles divergentes dans Power BI',
        },
        {
            'theme': 'eligibilite_priorisation_2016',
            'constat': f"{len(priority_snapshot)} pays eligibles, puis {len(priority_snapshot_focus)} pays apres filtre population >= 500k",
            'decision': 'le snapshot analytique conserve les pays classes ; le focus dashboard limite la restitution aux unites les plus actionnables',
        },
    ]
)

afficher_notebook(arbitrage_priorisation, lignes=len(arbitrage_priorisation))

,theme,constat,decision
0,valeurs_manquantes_snapshot_2016,2 pays non classes sur 237 observations 2016,pas d'imputation automatique ; exclusion des p...
1,couverture_service_gere_securise_2016,139 valeurs manquantes sur le service gere en ...,le score de modernisation reste calculable seu...
2,audit_population_china,China est egal a la somme China mainland + Hon...,conserver l'information dans les marts mais in...
3,normalisation_libelles_exports_pbi,les exports PBI normalisent China mainland en ...,centraliser cette regle dans le script de reco...
4,eligibilite_priorisation_2016,"235 pays eligibles, puis 173 pays apres filtre...",le snapshot analytique conserve les pays class...


### 7.2 Arbitrages métier pour la priorisation

Les résultats précédents imposent quatre arbitrages explicites pour la suite du pipeline et du dashboard :

1. les valeurs manquantes ne sont pas imputées automatiquement dans ce notebook ; on préfère conserver la trace des absences plutôt que fabriquer un signal artificiel ;
2. la priorisation 2016 retient les pays classés sur au moins un score métier, puis applique un filtre de focalisation population >= 500 000 habitants ;
3. le cas China est conservé dans les exports comme information métier, mais il ne doit jamais être additionné naïvement avec ses composantes dans une lecture agrégée ou un top pays ;
4. les exports Power BI normalisent désormais les libellés China mainland, China Taiwan Province of, China Hong Kong SAR et China Macao SAR en Chine, Taiwan, Hong Kong et Macao, avec une géographie homogénéisée sur Asia / Eastern Asia pour les trois composantes non continentales.

## Partie 8 - Exports CSV pour le pipeline et le dashboard

 > Le notebook produit quatre sorties : référentiel pays-régions, table longue normalisée, table large dashboard et snapshot de priorisation.

### 8.1 Fichiers exportés

 > Ces exports sont pensés pour être réutilisés dans le pipeline Python et dans le futur modèle en étoile du dashboard.

In [ ]:
CSV_EN_PATH = PROCESSED_DATA_PATH / 'csv_enrichis' / 'en'
CSV_FR_PATH = PROCESSED_DATA_PATH / 'csv_enrichis' / 'fr'
PBI_STAR_PATH = PROCESSED_DATA_PATH / 'pbi_star'

CSV_EN_PATH.mkdir(parents=True, exist_ok=True)
CSV_FR_PATH.mkdir(parents=True, exist_ok=True)
PBI_STAR_PATH.mkdir(parents=True, exist_ok=True)

csv_export_separator = ';'

french_export_columns = {
    'country': 'pays',
    'country_clean': 'pays',
    'region': 'region',
    'year': 'annee',
    'granularity': 'granularite',
    'value': 'valeur',
    'indicator': 'indicateur',
    'unit': 'unite',
    'source': 'source',
    'scope_note': 'note_portee',
    **EXPORT_PBI_FR,
    'rural_share_pct': 'part_rurale_pct',
    'basic_minus_safe_gap_pct': 'ecart_basique_safely_managed_pct',
    'risk_low_basic': 'risque_faible_acces_basique',
    'risk_low_safe': 'risque_faible_acces_safely_managed',
    'risk_high_gap': 'risque_ecart_qualite_eleve',
    'risk_high_mortality': 'risque_mortalite_elevee',
    'risk_low_stability': 'risque_faible_stabilite',
    'risk_high_rurality': 'risque_ruralite_elevee',
    'gov_effectiveness_score': 'score_effectivite_gouvernementale',
    'score_creation_services': 'score_creation_services',
    'score_modernisation_services': 'score_modernisation_services',
    'score_gouvernance': 'score_gouvernance',
    'action_prioritaire': 'action_prioritaire',
}

def exporter_version_fr(dataframe, output_path):
    dataframe.rename(columns=french_export_columns).to_csv(
        output_path, index=False, sep=csv_export_separator
    )

dashboard_water_star = dashboard_water.copy()
dashboard_water_star['action_prioritaire'] = dashboard_water_star['action_prioritaire'].fillna('Non classe')

dim_pays_star = (
    dashboard_water_star[['country', 'region']]
    .drop_duplicates()
    .sort_values(['region', 'country'])
    .reset_index(drop=True)
    .assign(cle_pays=lambda df: df.index + 1)
    [['cle_pays', 'country', 'region']]
 )

dim_annee_star = (
    dashboard_water_star[['year']]
    .drop_duplicates()
    .sort_values('year')
    .reset_index(drop=True)
    .assign(cle_annee=lambda df: df['year'].astype(int))
    [['cle_annee', 'year']]
 )

dim_action_star = (
    dashboard_water_star[['action_prioritaire']]
    .drop_duplicates()
    .sort_values('action_prioritaire')
    .reset_index(drop=True)
    .assign(cle_action=lambda df: df.index + 1)
    [['cle_action', 'action_prioritaire']]
 )
dim_action_star['domaine_action'] = dim_action_star['action_prioritaire'].map({
    'Créer ou étendre les services d\'eau': 'Creation de services',
    'Moderniser les services d\'eau': 'Modernisation des services',
    'Renforcer la gouvernance politique': 'Gouvernance',
    'Non classe': 'Hors priorisation',
}).fillna('Hors priorisation')
dim_action_star = dim_action_star[['cle_action', 'action_prioritaire', 'domaine_action']]

fact_dashboard_star = (
    dashboard_water_star
    .merge(dim_pays_star, on=['country', 'region'], how='left')
    .merge(dim_annee_star, on='year', how='left')
    .merge(dim_action_star, on='action_prioritaire', how='left')
    .rename(columns={'country': 'pays', 'year': 'annee'})
)
fact_dashboard_star = fact_dashboard_star[[
    'cle_pays',
    'cle_annee',
    'cle_action',
    'pays',
    'region',
    'annee',
    'population_total_people',
    'population_rural_people',
    'political_stability',
    'basic_drinking_water_pct',
    'safely_managed_drinking_water_pct',
    'wash_mortality_rate_per_100k',
    'wash_deaths',
    'rural_share_pct',
    'basic_minus_safe_gap_pct',
    'risk_low_basic',
    'risk_low_safe',
    'risk_high_gap',
    'risk_high_mortality',
    'risk_low_stability',
    'risk_high_rurality',
    'gov_effectiveness_score',
    'score_creation_services',
    'score_modernisation_services',
    'score_gouvernance',
    'action_prioritaire',
]]

export_paths = {
    'country_region_reference': CSV_EN_PATH / 'country_region_reference.csv',
    'water_indicators_long': CSV_EN_PATH / 'water_indicators_long.csv',
    'dashboard_water_country_year': CSV_EN_PATH / 'dashboard_water_country_year.csv',
    'priority_snapshot': CSV_EN_PATH / f'priority_snapshot_{analysis_year}.csv',
    'priority_snapshot_focus': CSV_EN_PATH / f'priority_snapshot_focus_{analysis_year}.csv',
    'country_region_reference_fr': CSV_FR_PATH / 'country_region_reference_fr.csv',
    'water_indicators_long_fr': CSV_FR_PATH / 'water_indicators_long_fr.csv',
    'dashboard_water_country_year_fr': CSV_FR_PATH / 'dashboard_water_country_year_fr.csv',
    'priority_snapshot_fr': CSV_FR_PATH / f'priority_snapshot_{analysis_year}_fr.csv',
    'priority_snapshot_focus_fr': CSV_FR_PATH / f'priority_snapshot_focus_{analysis_year}_fr.csv',
    'dim_pays_star_fr': PBI_STAR_PATH / 'dim_pays_star_fr.csv',
    'dim_annee_star_fr': PBI_STAR_PATH / 'dim_annee_star_fr.csv',
    'dim_action_star_fr': PBI_STAR_PATH / 'dim_action_star_fr.csv',
    'fact_dashboard_star_fr': PBI_STAR_PATH / 'fact_dashboard_star_fr.csv',
}

country_region_reference.sort_values(['region', 'country_clean']).to_csv(
    export_paths['country_region_reference'], index=False, sep=csv_export_separator
)
normalized_long.to_csv(export_paths['water_indicators_long'], index=False, sep=csv_export_separator)
dashboard_water.to_csv(export_paths['dashboard_water_country_year'], index=False, sep=csv_export_separator)
priority_snapshot.to_csv(export_paths['priority_snapshot'], index=False, sep=csv_export_separator)
priority_snapshot_focus.to_csv(export_paths['priority_snapshot_focus'], index=False, sep=csv_export_separator)

exporter_version_fr(
    country_region_reference.sort_values(['region', 'country_clean']),
    export_paths['country_region_reference_fr'],
)
exporter_version_fr(normalized_long, export_paths['water_indicators_long_fr'])
exporter_version_fr(dashboard_water, export_paths['dashboard_water_country_year_fr'])
exporter_version_fr(priority_snapshot, export_paths['priority_snapshot_fr'])
exporter_version_fr(priority_snapshot_focus, export_paths['priority_snapshot_focus_fr'])

dim_pays_star.rename(columns={'country': 'pays', 'region': 'region'}).to_csv(
    export_paths['dim_pays_star_fr'], index=False, sep=csv_export_separator
)
dim_annee_star.rename(columns={'year': 'annee'}).to_csv(
    export_paths['dim_annee_star_fr'], index=False, sep=csv_export_separator
)
dim_action_star.to_csv(
    export_paths['dim_action_star_fr'], index=False, sep=csv_export_separator
)
exporter_version_fr(fact_dashboard_star, export_paths['fact_dashboard_star_fr'])

In [70]:
export_summary = pd.DataFrame(
    [
        {'dataset': 'country_region_reference', 'rows': len(country_region_reference), 'path': str(export_paths['country_region_reference'])},
        {'dataset': 'water_indicators_long', 'rows': len(normalized_long), 'path': str(export_paths['water_indicators_long'])},
        {'dataset': 'dashboard_water_country_year', 'rows': len(dashboard_water), 'path': str(export_paths['dashboard_water_country_year'])},
        {'dataset': 'priority_snapshot', 'rows': len(priority_snapshot), 'path': str(export_paths['priority_snapshot'])},
        {'dataset': 'priority_snapshot_focus', 'rows': len(priority_snapshot_focus), 'path': str(export_paths['priority_snapshot_focus'])},
        {'dataset': 'country_region_reference_fr', 'rows': len(country_region_reference), 'path': str(export_paths['country_region_reference_fr'])},
        {'dataset': 'water_indicators_long_fr', 'rows': len(normalized_long), 'path': str(export_paths['water_indicators_long_fr'])},
        {'dataset': 'dashboard_water_country_year_fr', 'rows': len(dashboard_water), 'path': str(export_paths['dashboard_water_country_year_fr'])},
        {'dataset': 'priority_snapshot_fr', 'rows': len(priority_snapshot), 'path': str(export_paths['priority_snapshot_fr'])},
        {'dataset': 'priority_snapshot_focus_fr', 'rows': len(priority_snapshot_focus), 'path': str(export_paths['priority_snapshot_focus_fr'])},
        {'dataset': 'dim_pays_star_fr', 'rows': len(dim_pays_star), 'path': str(export_paths['dim_pays_star_fr'])},
        {'dataset': 'dim_annee_star_fr', 'rows': len(dim_annee_star), 'path': str(export_paths['dim_annee_star_fr'])},
        {'dataset': 'dim_action_star_fr', 'rows': len(dim_action_star), 'path': str(export_paths['dim_action_star_fr'])},
        {'dataset': 'fact_dashboard_star_fr', 'rows': len(fact_dashboard_star), 'path': str(export_paths['fact_dashboard_star_fr'])},
    ]
)

afficher_notebook(export_summary, lignes=len(export_summary))

,dataset,lignes,path
0,country_region_reference,240,C:\Users\feria\Documents\P10\data\processed\cs...
1,water_indicators_long,46490,C:\Users\feria\Documents\P10\data\processed\cs...
2,dashboard_water_country_year,4466,C:\Users\feria\Documents\P10\data\processed\cs...
3,priority_snapshot,235,C:\Users\feria\Documents\P10\data\processed\cs...
4,priority_snapshot_focus,173,C:\Users\feria\Documents\P10\data\processed\cs...
5,country_region_reference_fr,240,C:\Users\feria\Documents\P10\data\processed\cs...
6,water_indicators_long_fr,46490,C:\Users\feria\Documents\P10\data\processed\cs...
7,dashboard_water_country_year_fr,4466,C:\Users\feria\Documents\P10\data\processed\cs...
8,priority_snapshot_fr,235,C:\Users\feria\Documents\P10\data\processed\cs...
9,priority_snapshot_focus_fr,173,C:\Users\feria\Documents\P10\data\processed\cs...


## Partie 9 - Insights et trame narrative du dashboard

 > Le but n'est pas seulement d'afficher des indicateurs, mais de relier chaque pays à une recommandation d'action prioritaire.

In [65]:
regional_summary = (
    priority_snapshot_focus.groupby("region")
    .agg(
        countries=("country", "nunique"),
        avg_basic_water_pct=("basic_drinking_water_pct", "mean"),
        avg_safe_water_pct=("safely_managed_drinking_water_pct", "mean"),
        avg_political_stability=("political_stability", "mean"),
        avg_wash_mortality_rate=("wash_mortality_rate_per_100k", "mean"),
        avg_creation_score=("score_creation_services", "mean"),
        avg_modernisation_score=("score_modernisation_services", "mean"),
        avg_gouvernance_score=("score_gouvernance", "mean"),
    )
    .round(2)
    .sort_values("avg_gouvernance_score", ascending=False)
    .reset_index()
)

top_creation = priority_snapshot_focus.sort_values("score_creation_services", ascending=False)[
    ["country", "region", "score_creation_services", "basic_drinking_water_pct", "rural_share_pct"]
]
top_modernisation = priority_snapshot_focus.sort_values("score_modernisation_services", ascending=False)[
    ["country", "region", "score_modernisation_services", "basic_minus_safe_gap_pct", "safely_managed_drinking_water_pct"]
]
top_gouvernance = priority_snapshot_focus.sort_values("score_gouvernance", ascending=False)[
    ["country", "region", "score_gouvernance", "political_stability", "wash_mortality_rate_per_100k"]
]

afficher_notebook(regional_summary)
afficher_notebook(top_creation)
afficher_notebook(top_modernisation)
afficher_notebook(top_gouvernance)

,region,nb_pays,avg_basic_water_pct,avg_safe_water_pct,avg_political_stability,avg_wash_mortality_rate,avg_creation_score,avg_modernisation_score,avg_gouvernance_score
0,Africa,47,65.57,23.31,-0.59,40.40,43.96,43.09,46.20
1,Eastern Mediterranean,22,88.27,79.53,-1.26,9.23,17.95,17.50,41.56
2,South-East Asia,10,90.35,46.33,-0.48,9.00,25.37,26.26,34.29
3,Americas,27,94.59,72.82,0.06,2.50,11.56,12.54,24.28
4,Western Pacific,19,87.83,66.76,0.52,3.57,18.23,19.60,20.90
5,Europe,48,97.37,90.12,0.22,0.35,10.60,8.38,18.90


,pays,region,score_creation_services,acces_eau_potable_pct,part_rurale_pct
765,Chad,Africa,80.033254,38.85259,76.743592
3767,South Sudan,Africa,73.638902,40.81390,91.351558
2901,Niger,Africa,68.454585,49.50156,83.243625
746,Central African Republic,Africa,66.620769,46.33376,60.127409
1362,Ethiopia,Africa,65.857546,40.04142,79.205829
3729,Somalia,Eastern Mediterranean,64.300440,50.81511,56.708131
1305,Eritrea,Africa,62.707321,51.84972,89.763537
632,Burundi,Africa,62.480822,60.20415,87.913429
1134,Democratic Republic of the Congo,Africa,61.988033,42.97677,56.654914
613,Burkina Faso,Africa,60.757072,48.26772,71.866698


,pays,region,score_modernisation_services,ecart_basique_vs_gere_securise_pct,service_eau_gere_securise_pct
765,Chad,Africa,100.000000,NaN,NaN
3729,Somalia,Eastern Mediterranean,85.680581,NaN,NaN
3626,Sierra Leone,Africa,85.269905,50.02917,9.53397
746,Central African Republic,Africa,81.264262,NaN,NaN
2920,Nigeria,Africa,77.924590,50.12199,19.90354
2198,Lao People's Democratic Republic,Western Pacific,76.067516,64.25883,15.68307
4159,Uganda,Africa,71.701163,40.71410,6.95153
2787,Nepal,South-East Asia,70.815968,61.28821,27.05820
2901,Niger,Africa,70.082044,NaN,NaN
2445,Mali,Africa,69.989444,NaN,NaN


,pays,region,score_gouvernance,political_stability,mortalite_wash_pour_100k
3729,Somalia,Eastern Mediterranean,83.384953,-2.36,86.57523
765,Chad,Africa,76.207386,-1.30,101.04312
3767,South Sudan,Africa,75.451584,-2.42,63.28665
746,Central African Republic,Africa,74.981674,-1.79,82.11312
3053,Palestine,Eastern Mediterranean,74.810606,-1.98,NaN
2920,Nigeria,Africa,73.612828,-1.88,68.62818
1134,Democratic Republic of the Congo,Africa,71.892094,-2.23,59.75592
632,Burundi,Africa,70.909687,-1.97,65.40117
2445,Mali,Africa,68.741307,-1.62,70.72140
1362,Ethiopia,Africa,65.631080,-1.62,43.66399


### 9.1 Angles narratifs recommandés

 > Le dashboard peut maintenant raconter trois histoires complémentaires : où créer, où moderniser, et où renforcer la gouvernance.

In [66]:
top_creation_names = ", ".join(top_creation["country"].head(5).tolist())
top_modernisation_names = ", ".join(top_modernisation["country"].head(5).tolist())
top_gouvernance_names = ", ".join(top_gouvernance["country"].head(5).tolist())
highest_governance_region = regional_summary.iloc[0]["region"]

print(f"1. Création de services : parmi les pays de plus de 500 000 habitants, les priorités en {analysis_year} sont {top_creation_names}.")
print(f"2. Modernisation : les situations les plus critiques concernent {top_modernisation_names}.")
print(f"3. Gouvernance : les signaux les plus forts concernent {top_gouvernance_names}.")
print(f"4. Lecture régionale : la région avec le score moyen de gouvernance le plus élevé est {highest_governance_region}.")
print("5. Prudence méthodologique : l'indicateur WASH mesure une vulnérabilité sanitaire combinée eau + assainissement + hygiène. Il doit donc rester un proxy d'alerte, pas une preuve causale de l'eau potable seule.")

1. Création de services : parmi les pays de plus de 500 000 habitants, les priorités en 2016 sont Chad, South Sudan, Niger, Central African Republic, Ethiopia.
2. Modernisation : les situations les plus critiques concernent Chad, Somalia, Sierra Leone, Central African Republic, Nigeria.
3. Gouvernance : les signaux les plus forts concernent Somalia, Chad, South Sudan, Central African Republic, Palestine.
4. Lecture régionale : la région avec le score moyen de gouvernance le plus élevé est Africa.
5. Prudence méthodologique : l'indicateur WASH mesure une vulnérabilité sanitaire combinée eau + assainissement + hygiène. Il doit donc rester un proxy d'alerte, pas une preuve causale de l'eau potable seule.


## Partie 10 - Conclusion pipeline

 > Ce notebook prépare les fondations du pipeline de données. La logique retenue est la suivante :

 > 1. les données brutes sont inspectées et standardisées ;

 > 2. les pays sont harmonisés et rattachés à une macro-région ;

 > 3. la population est convertie en habitants ;

 > 4. les sources sont normalisées dans une table longue, puis projetées dans une table large pays-année ;

 > 5. la mortalité WASH est conservée comme référence 2016 pour enrichir la priorisation, sans prétendre mesurer une évolution temporelle.

Point méthodologique à conserver pour la restitution : `wash_mortality_rate_per_100k` est un taux standardisé pour 100 000 habitants et reste donc naturellement décimal. `wash_deaths` correspond à un nombre de décès estimé par modélisation sanitaire WASH ; il peut donc aussi présenter des décimales sans incohérence, car il ne s'agit pas d'un décompte administratif exhaustif.

In [71]:
from pathlib import Path
import pandas as pd

EXPORT_FILES = [
    PROJECT_ROOT / 'data' / 'processed' / 'csv_enrichis' / 'en' / 'dashboard_water_country_year.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'csv_enrichis' / 'en' / 'priority_snapshot_2016.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'csv_enrichis' / 'en' / 'priority_snapshot_focus_2016.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'csv_enrichis' / 'fr' / 'dashboard_water_country_year_fr.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'csv_enrichis' / 'fr' / 'priority_snapshot_2016_fr.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'csv_enrichis' / 'fr' / 'priority_snapshot_focus_2016_fr.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'pbi_star' / 'fact_dashboard_star_fr.csv',
]

POPULATION_COLUMNS = [
    'population_total_people',
    'population_rural_people',
    'population_totale',
    'population_rurale',
]

for export_file in EXPORT_FILES:
    if not export_file.exists():
        print(f"Fichier absent : {export_file.name}")
        continue

    export_df = pd.read_csv(export_file, sep=';')
    converted_columns = []

    for column in POPULATION_COLUMNS:
        if column in export_df.columns:
            export_df[column] = pd.to_numeric(export_df[column], errors='coerce').round().astype('Int64')
            converted_columns.append(column)

    export_df.to_csv(export_file, sep=';', index=False)
    print(f"{export_file.name} -> colonnes converties : {', '.join(converted_columns)}")

dashboard_water_country_year.csv -> colonnes converties : population_total_people, population_rural_people
priority_snapshot_2016.csv -> colonnes converties : population_total_people, population_rural_people
priority_snapshot_focus_2016.csv -> colonnes converties : population_total_people, population_rural_people
dashboard_water_country_year_fr.csv -> colonnes converties : population_totale, population_rurale
priority_snapshot_2016_fr.csv -> colonnes converties : population_totale, population_rurale
priority_snapshot_focus_2016_fr.csv -> colonnes converties : population_totale, population_rurale
fact_dashboard_star_fr.csv -> colonnes converties : population_totale, population_rurale
